# Experiment 1 — Do score-derived tangent directions explain black-box transferability?

**Standalone, reproducible Google Colab.** Reuses the *exact original* pretrained class-conditional DDPM score-Jacobian spectral filter from `MNIST_Digit1_Final_Geometry.py`. It trains four independent MNIST classifiers, estimates the original score-Jacobian Hill filter applied directly to classifier gradients, plus a finite filtered-probe subspace diagnostic, and compares **gradient alignment**, **gradient magnitude**, and **one-step transfer**.

### Scientific hypothesis (not assumed true)

> Gradients of independently trained classifiers may align better in geometry-supported directions than in arbitrary subspaces, and this may translate into higher target-model loss or attack transfer.

The experiment explicitly permits a **negative result**. Tangency does **not** guarantee semantic preservation or transferability.

### Critical distinctions

- `score_t5` / `score_t100`: a **rank-limited span of spectral-filtered random probes**. This is a heuristic *candidate tangent subspace*, **not the true tangent space**, not an exact 784×784 projector, and not an official LHSD estimator.
- This experiment uses **precommitted fixed scales** (`t=5`, `t=100`) to isolate a mechanism. **It does not claim to implement the previous FIT-ranked calibrated selector**. That can be tested later after the mechanism is established.
- **Target classifier input gradients are computed only for the offline mechanism analysis.** They do not enter perturbation generation or hyperparameter choice. Transfer attacks use the current surrogate's gradient only. Thus transfer generation is target-query-free; this is **not** a strictly black-box-only diagnostic experiment.
- All comparisons include **random subspaces** and **training-data local PCA** controls matched to a score-derived energy-equivalent rank; the finite probe-bank comparison uses exact matched rank. Report projected gradient norm as well as cosine: low-rank projections can artificially inflate cosine.
- Prototype is restricted to **MNIST digit 1** because the frozen diffusion score model is conditioned on class 1. Extend to other classes only after validating the mechanism.
- Transfer attacks use **identical post-clipping L2 budgets** through a deterministic input-only line search, and are **single-step probes**, not a trained generator or a state-of-the-art attack benchmark.

**Outputs:** model manifest, geometry diagnostics, pairwise gradient alignment, paired bootstrap confidence intervals clustered by image, target transfer matrix, three primary figures, protocol metadata, and proposal-safe readme.

In [ ]:
%pip -q install "diffusers>=0.29,<1" "huggingface_hub>=0.25" safetensors matplotlib pandas scikit-learn

In [ ]:
import os, json, math, hashlib, time, random, shutil, warnings
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from torchvision.datasets import MNIST

# Google Drive is optional outside Colab. All output paths are explicit.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except ImportError:
    DRIVE_ROOT = Path('/mnt/data')

RUN_MODE='pilot'                 # change to 'main' after reviewing the pilot; do not tune from target results
USE_DRIVE=True
SEED=20261103
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if RUN_MODE=='pilot':
    N_DIGIT1=24
    TRAIN_EPOCHS=3
    TRAIN_LIMIT=40000
    FILTER_PROBES=8
    FILTER_LANCZOS=8
else:
    N_DIGIT1=160
    TRAIN_EPOCHS=6
    TRAIN_LIMIT=55000
    FILTER_PROBES=16
    FILTER_LANCZOS=10

FIXED_TIMESTEPS=(5,100)
N_RANDOM_CONTROLS=3
PCA_NEIGHBORS=64
ATTACK_EPS_L2=2.0           # L2 measured in [-1,1] pixels; equals 1.0 in [0,1] coordinates
MIN_GRAD_NORM=1e-9
MIN_PROJECTED_RATIO=1e-4
BOOTSTRAP_REPS=2000
DIRECT_FILTER_GRADIENTS=True
DATA_ROOT=Path('/content/transfer_alignment_mnist' if Path('/content').exists() else '/mnt/data/transfer_alignment_mnist')
OUT_ROOT=DRIVE_ROOT/'MNIST_Tangent_Transfer_Alignment_Exp1' if USE_DRIVE else DATA_ROOT/'results'
OUT_ROOT.mkdir(parents=True,exist_ok=True)
CACHE=OUT_ROOT/'cache'; CACHE.mkdir(exist_ok=True)
FIGS=OUT_ROOT/'figures';FIGS.mkdir(exist_ok=True)
print('DEVICE:', DEVICE,'| MODE:', RUN_MODE, '| output:',OUT_ROOT)

In [ ]:
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
seed_all(SEED)
torch.set_num_threads(min(6,os.cpu_count() or 2))
protocol=dict(mode=RUN_MODE,seed=SEED,geometry_timesteps=list(FIXED_TIMESTEPS),
  number_requested=N_DIGIT1,train_epochs=TRAIN_EPOCHS,train_limit=TRAIN_LIMIT,
  score_model='harveymannering/mnist-ddpm',filter_probes=FILTER_PROBES,
  filter_lanczos=FILTER_LANCZOS,filter_cutoff=.1,hill_power=4,
  random_control_repeats=N_RANDOM_CONTROLS,pca_neighbors=PCA_NEIGHBORS,
  attack_eps_L2_minus1_plus1=ATTACK_EPS_L2,
  selection='PRECOMMITTED FIXED TIMESTEPS, NOT FIT-RANKED CALIBRATED SELECTION',
  loss='logsumexp(nontrue_logits) - true_logit',
  target_use='alignment diagnostics white-box only; attack generation uses surrogate gradient exclusively',
  interpretation='score-filtered low-rank span is a candidate, not certified tangent geometry')
(OUT_ROOT/'EXPERIMENT_PROTOCOL.json').write_text(json.dumps(protocol,indent=2))

## 1. Load MNIST and freeze the evaluation design

Classifiers train on the **MNIST training split**. Test images come from the separate **MNIST test split**, restricted to label `1`. The test-index selection seed is fixed before running. We exclude known prior geometry demo indices and, when available, the repair notebook's recorded FIT/CAL indices. All reported transfer rates use examples correctly classified **before** the attack by every evaluated model; filtering on clean correctness is part of the predefined evaluation protocol, not attack tuning.

In [ ]:
train_ds=MNIST(str(DATA_ROOT),train=True,download=True)
test_ds=MNIST(str(DATA_ROOT),train=False,download=True)
assert train_ds.data.shape[1:]==(28,28)

# Exclude all explicitly recorded new scale-repair FIT/CAL indices when available.
excluded={2,409,2816,7812}
repair_protocol=DRIVE_ROOT/'MNIST_scale_repair'/'REPAIR_PROTOCOL.json'
if repair_protocol.exists():
    rp=json.loads(repair_protocol.read_text())
    for k in ('new_fit_indices','new_calibration_indices','new_cal_indices'):
        if k in rp: excluded.update(int(v) for v in rp[k])
    print('Excluded prior FIT/CAL indexes from',repair_protocol)
else:
    print('NOTE: Existing repair split metadata not found; cannot certify no reuse of prior geometry calibration samples.')
ones=torch.where(test_ds.targets==1)[0].tolist()
ones=[i for i in ones if i not in excluded]
rng=np.random.default_rng(SEED+71)
image_indices=rng.permutation(ones)[:N_DIGIT1].astype(int).tolist()
assert len(image_indices)==N_DIGIT1
(OUT_ROOT/'EVALUATION_INDICES.json').write_text(json.dumps(image_indices,indent=2))
print('Precommitted evaluation index count:',len(image_indices),'first:',image_indices[:12])

# Standard classifier input is [-1,1], matching the score network's MNIST convention.
def image_batch(index_list):
    a=test_ds.data[index_list].float()/127.5-1.0
    return a[:,None,:,:].contiguous()
all_x=image_batch(image_indices)
assert bool(((all_x>=-1)&(all_x<=1)).all())
train_norm=(train_ds.data.float()/127.5-1.0).unsqueeze(1)
train_targets=train_ds.targets.long()

## 2. Train independent classifiers (and cache them)

Four classifiers: two separate CNN initializations, an MLP, and a residual CNN. No target model is used to choose a scale, geometry vector, attack direction, or perturbation budget. Cache checks ensure the same training configuration is not accidentally mixed with an earlier run.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(784,256),nn.ReLU(),
             nn.Linear(256,128),nn.ReLU(),nn.Linear(128,10))
    def forward(self,x):return self.net(x)

class ConvBlock(nn.Module):
    def __init__(self,ci,co):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(ci,co,3,padding=1,bias=False),
              nn.BatchNorm2d(co),nn.ReLU(inplace=False))
    def forward(self,x):return self.net(x)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(ConvBlock(1,24),nn.MaxPool2d(2),
            ConvBlock(24,48),nn.MaxPool2d(2),nn.Flatten(),
            nn.Linear(48*7*7,96),nn.ReLU(),nn.Linear(96,10))
    def forward(self,x):return self.net(x)

class Residual(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.conv=nn.Sequential(ConvBlock(c,c),nn.Conv2d(c,c,3,padding=1,bias=False),nn.BatchNorm2d(c))
    def forward(self,x):return F.relu(x+self.conv(x))

class TinyResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(ConvBlock(1,24),Residual(24),nn.MaxPool2d(2),
             ConvBlock(24,48),Residual(48),nn.MaxPool2d(2),
             ConvBlock(48,64),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(64,10))
    def forward(self,x):return self.net(x)

MODEL_SPECS=[('cnn_A','smallcnn',SEED+10),('cnn_B','smallcnn',SEED+11),
             ('mlp','mlp',SEED+12),('resnet_tiny','resnet_tiny',SEED+13)]
def create_classifier(arch):
    return {'smallcnn':SmallCNN,'mlp':MLP,'resnet_tiny':TinyResNet}[arch]()

# Fixed train/validation split uses MNIST train only.
rand_idx=torch.randperm(len(train_ds),generator=torch.Generator().manual_seed(SEED+91)).tolist()
val_idx=rand_idx[-5000:]
tr_idx=rand_idx[:TRAIN_LIMIT]
train_loader=DataLoader(torch.utils.data.TensorDataset(train_norm[tr_idx],train_targets[tr_idx]),
        batch_size=256,shuffle=True,generator=torch.Generator().manual_seed(SEED+92),num_workers=0)
val_loader=DataLoader(torch.utils.data.TensorDataset(train_norm[val_idx],train_targets[val_idx]),
        batch_size=512,shuffle=False,num_workers=0)

@torch.no_grad()
def eval_accuracy(model,loader):
    model.eval();hit=total=0
    for x,y in loader:
        z=model(x.to(DEVICE));hit+=int((z.argmax(1)==y.to(DEVICE)).sum());total+=len(y)
    return hit/total

def fit_or_load_classifier(name,arch,init_seed):
    filename=CACHE/f'{name}_{RUN_MODE}.pt'
    expected=dict(name=name,arch=arch,init_seed=init_seed,train_limit=TRAIN_LIMIT,
                  epochs=TRAIN_EPOCHS,split_seed=SEED+91)
    seed_all(init_seed)
    model=create_classifier(arch).to(DEVICE)
    if filename.exists():
        packet=torch.load(filename,map_location='cpu',weights_only=True)
        assert packet['config']==expected, f'Wrong cached training settings for {name}'
        model.load_state_dict(packet['state_dict']);model.eval()
        return model,float(packet['val_accuracy']),filename
    optim=torch.optim.Adam(model.parameters(),lr=1e-3)
    best_acc=-1.;best_state=None
    for epoch in range(TRAIN_EPOCHS):
        model.train();total_loss=0.
        for images,labels in train_loader:
            images=images.to(DEVICE);labels=labels.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            z=model(images);loss=F.cross_entropy(z,labels)
            loss.backward();optim.step()
            total_loss+=float(loss.detach())*len(labels)
        score=eval_accuracy(model,val_loader)
        print(f'{name:12s} | epoch {epoch+1}/{TRAIN_EPOCHS} | val accuracy {score:.4f}')
        if score>best_acc:
            best_acc=score
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state);model.eval()
    torch.save(dict(config=expected,state_dict=best_state,val_accuracy=best_acc),filename)
    return model,best_acc,filename

models={};manifest=[]
for name,arch,init_seed in MODEL_SPECS:
    net,acc,path=fit_or_load_classifier(name,arch,init_seed)
    for p in net.parameters():p.requires_grad_(False)
    models[name]=net
    manifest.append(dict(model=name,arch=arch,seed=init_seed,val_accuracy=acc,checkpoint=str(path)))
pd.DataFrame(manifest).to_csv(OUT_ROOT/'CLASSIFIER_MANIFEST.csv',index=False)
display(pd.DataFrame(manifest))

In [ ]:
# Clean correctness is determined once for the fixed evaluation indices.
@torch.no_grad()
def batch_clean_logits(model,x,chunk=64):
    model.eval();outputs=[]
    for i in range(0,len(x),chunk):
        outputs.append(model(x[i:i+chunk].to(DEVICE)).cpu())
    return torch.cat(outputs)
logits_clean={name:batch_clean_logits(net,all_x) for name,net in models.items()}
keep=np.ones(len(image_indices),dtype=bool)
clean_acc={}
for name,z in logits_clean.items():
    correct=(z.argmax(1)==1).numpy()
    clean_acc[name]=float(correct.mean());keep &= correct
print('Clean accuracy on precommitted digit-1 evaluation images:',clean_acc)
print('Correct for ALL four classifiers:',int(keep.sum()),'/',len(keep))
assert int(keep.sum())>=5, 'Too few jointly clean-correct images. Increase train epochs before any attack evaluation; log this change.'
active_indices=np.array(image_indices)[keep].tolist()
X=all_x[keep].clone()
Y=torch.ones(len(X),dtype=torch.long)
clean_selected={name:z[keep] for name,z in logits_clean.items()}
(OUT_ROOT/'JOINTLY_CORRECT_EVALUATION_INDICES.json').write_text(json.dumps(active_indices,indent=2))

## 3. Input gradients — fixed shared loss for all classifiers

For each classifier, use the **same smooth untargeted margin**:

\[\ell_f(x,y)=\log\sum_{j
e y}\exp z_{f,j}(x)-z_{f,y}(x).\]

This avoids comparing gradients of different manually selected target classes. For the **mechanism** we compute all classifiers' input gradients offline. No target gradient will be used for the attack step.

In [ ]:
def margin_objective(logits,labels):
    y=labels.long().to(logits.device)
    masked=logits.masked_fill(F.one_hot(y,logits.shape[1]).bool(),float('-inf'))
    log_nontrue=torch.logsumexp(masked,dim=1)
    true=logits.gather(1,y[:,None]).squeeze(1)
    return log_nontrue-true

def input_gradients(model,x,y,batch_size=16):
    model.eval();grad_list=[]
    for i in range(0,len(x),batch_size):
        xb=x[i:i+batch_size].to(DEVICE).detach().requires_grad_(True)
        obj=margin_objective(model(xb),y[i:i+batch_size].to(DEVICE))
        g=torch.autograd.grad(obj.sum(),xb)[0]
        grad_list.append(g.detach().flatten(1).cpu())
    return torch.cat(grad_list)

GRAD_CACHE=CACHE/f'gradient_cache_{RUN_MODE}.pt'
identity=dict(indices=active_indices,models={name:sp[2] for name,sp in zip(models,MODEL_SPECS)},loss='logsumexp_other_minus_true',epochs=TRAIN_EPOCHS)
if GRAD_CACHE.exists():
    pack=torch.load(GRAD_CACHE,map_location='cpu',weights_only=True)
    assert pack['identity']==identity,'Cached gradient mismatch'
    grads=pack['grads'];print('Reusing gradient cache')
else:
    grads={name:input_gradients(model,X,Y) for name,model in models.items()}
    torch.save(dict(identity=identity,grads=grads),GRAD_CACHE)
for name in grads:
    print(name,'median norm',float(grads[name].norm(dim=1).median()))

## 4. Original score operator and Task 2 spectral filter

This cell embeds the frozen implementation from the earlier `MNIST_Digit1_Final_Geometry.py`. We do **not** substitute another geometry learner. All vectors are in the 784-dimensional MNIST pixel space.

The operator is the **symmetrized Jacobian of a DDPM epsilon-prediction network**, not a guaranteed exact Hessian of a conservative EBM.

In [ ]:
# Extracted without algorithmic modification from MNIST_Digit1_Final_Geometry.py.
# This is the ORIGINAL score-Jacobian operator, Lanczos and Hill filter.
import math, torch, numpy as np
from torch.nn import functional as F
MODEL_ID = 'harveymannering/mnist-ddpm'
SELECT = dict(transition=0.12, defect=0.24, cross=0.28, asym=0.06, range=0.12,
              spectral_cutoff=0.10, hill_power=4, min_run=2)
def alpha_cumprod(t):
    # Author's model card: linear DDPM betas 1e-4..0.02 over 1000 steps
    betas = torch.linspace(0.0001, 0.02, 1000, dtype=torch.float32)
    return float(torch.cumprod(1.0-betas, dim=0)[int(t)])


def model_load(device):
    try:
        from diffusers import UNet2DModel
    except Exception as ex:
        raise RuntimeError('Install diffusers and huggingface_hub first.') from ex
    net = UNet2DModel.from_pretrained(MODEL_ID, use_safetensors=True).to(device).eval()
    for p in net.parameters(): p.requires_grad_(False)
    assert net.config.num_class_embeds == 11, f'Expected conditional checkpoint (11 classes), got {net.config.num_class_embeds}'
    assert net.config.in_channels == 1 and net.config.sample_size == 28, 'Expected 1x28x28 images'
    return net



class ScoreOperator:
    """Symmetric part of normalized score Hessian Q, matrix-free."""
    def __init__(self, net, x, t, fd_eps=0.04):
        self.net = net
        self.t = int(t)
        self.ab = alpha_cumprod(t)
        self.sqrt_ab = math.sqrt(self.ab)
        self.scale = math.sqrt(1-self.ab)
        self.sigma = math.sqrt((1-self.ab)/self.ab)
        self.xt = self.sqrt_ab * x.detach().clone()
        self.device = x.device
        self.D = x.numel()
        self.fd_eps = float(fd_eps)
        self.image_forward_evals = 0
        self.backward_evals = 0
        self.sym_calls = 0
        self.raw_calls = 0
        self.adj_calls = 0

    def model_call(self, batch):
        ids=torch.full((batch.shape[0],),self.t,device=self.device,dtype=torch.long)
        labels=torch.ones(batch.shape[0],device=self.device,dtype=torch.long)
        return self.net(batch,ids,class_labels=labels).sample

    def q_raw(self, vector, h=None):
        """Q_raw v by forward differences of epsilon network (only forwards)."""
        self.raw_calls += 1
        v = vector.detach().to(self.xt).reshape_as(self.xt)
        n = float(torch.linalg.vector_norm(v))
        if n < 1e-15: return torch.zeros_like(vector)
        step = (h if h is not None else self.fd_eps)/n
        with torch.no_grad():
            xs=torch.cat([self.xt+step*v,self.xt-step*v],dim=0)
            out=self.model_call(xs)
            self.image_forward_evals += 2
            jv=(out[0:1]-out[1:2])/(2*step)
        return (self.scale*jv).reshape(-1)

    def q_adj(self, vector):
        """Q_raw^T v by first-order reverse-mode gradient, no double backward."""
        self.adj_calls += 1
        v=vector.detach().to(self.xt).reshape_as(self.xt)
        with torch.enable_grad():
            inp=self.xt.detach().clone().requires_grad_(True)
            out=self.model_call(inp)
            scalar=(out*v).sum()
            grad=torch.autograd.grad(scalar,inp,create_graph=False)[0]
        self.image_forward_evals += 1
        self.backward_evals += 1
        return (self.scale*grad).detach().reshape(-1)

    def q_sym(self, vector):
        self.sym_calls += 1
        return 0.5*(self.q_raw(vector)+self.q_adj(vector))

    def counters(self):
        return dict(sym_calls=self.sym_calls,fd_calls=self.raw_calls,
                    vjp_calls=self.adj_calls,model_forward_image_evals=self.image_forward_evals,
                    model_backward_evals=self.backward_evals)


class DiagOp:
    def __init__(self, diag):
        self.diag=torch.as_tensor(diag,dtype=torch.float32)
        self.D=self.diag.numel()
        self.device=self.diag.device
    def q_sym(self,v): return self.diag * v


def lanczos(op, v, nsteps=12):
    z=v.detach().flatten().float().to(op.device)
    mag=float(torch.linalg.vector_norm(z).item())
    if mag < 1e-14: raise ValueError('Near-zero Lanczos start vector')
    z=z/mag
    basis=[]; alphas=[];betas=[]
    prev=torch.zeros_like(z); prev_beta=0.0
    for it in range(min(nsteps,z.numel())):
        w=op.q_sym(z)
        a=float(torch.dot(z,w).item())
        w=w-a*z-prev_beta*prev
        # Two-pass full reorthogonalization for spectral numerical robustness.
        for _ in range(2):
            for q in basis+[z]:
                w=w-torch.dot(q,w)*q
        b=float(torch.linalg.vector_norm(w))
        basis.append(z)
        alphas.append(a)
        if b < 1e-8 or it==nsteps-1: break
        betas.append(b)
        prev=z;prev_beta=b;z=w/b
    T=np.diag(np.asarray(alphas,dtype=np.float64))
    for i,b in enumerate(betas):
        T[i,i+1]=b; T[i+1,i]=b
    eig,U=np.linalg.eigh(T)
    return eig,U,basis,mag


def hill(eig):
    return 1./(1.+(np.abs(eig)/SELECT['spectral_cutoff'])**SELECT['hill_power'])



def filter_apply(op, vector, steps):
    v=vector.detach().flatten().float().to(op.device)
    eig,U,basis,mag=lanczos(op,v,steps)
    coeff=(U * (hill(eig)*U[0,:])[None,:]).sum(axis=1)
    out=sum(float(c)*q for c,q in zip(coeff,basis))
    return mag*out

In [ ]:
# Analytic unit test for the original score-Jacobian normalization.
from types import SimpleNamespace
class LinearEpsMock(nn.Module):
    def __init__(self,diag,t):
        super().__init__()
        scale=math.sqrt(1-alpha_cumprod(t))
        self.register_buffer('weights',(diag/scale).reshape(1,1,28,28))
    def forward(self,x,t,class_labels):return SimpleNamespace(sample=x*self.weights)
test_diag=torch.cat([torch.zeros(96),torch.ones(688)])
mock=LinearEpsMock(test_diag,5)
op=ScoreOperator(mock,torch.zeros(1,1,28,28),5)
v=torch.randn(784,generator=torch.Generator().manual_seed(SEED))
assert torch.allclose(op.q_sym(v),test_diag*v,atol=1e-3,rtol=1e-3)
vec=torch.randn(784,generator=torch.Generator().manual_seed(SEED+8))
out=filter_apply(op,vec,steps=8)
exact=(1.-test_diag)*vec+hill(np.array([1.0]))[0]*test_diag*vec
assert torch.allclose(out,exact,atol=3e-3,rtol=3e-3)
print('PASS analytic score-Jacobian and Lanczos filter self-test.')

## 5. Build fixed-scale score-filtered subspaces (cache each image)

For each image and each `t∈{5,100}`, apply the **original Hill-filtered Lanczos map** to a frozen bank of isotropic random probes. Orthonormalize the retained vectors by SVD, using the same 0.15 singular-value threshold as Task 2. This creates an orthogonal projector onto the **retained finite-probe span**.

**Crucial:** This rank-limited bank is not the entire estimated tangent space and is not certified to contain the true manifold tangent. Record rank, filtering strength, and model-evaluation costs. All geometry is computed without classifier gradients.

**Primary diagnostic added:** we also apply the same original `filter_apply(op, gradient)` directly to every classifier input gradient. The latter is a **soft spectral filter**, not an exact projector. PCA/random control dimensions are matched to the filter’s `tr(H²)` estimate from independent random probes (capped by the local PCA neighborhood size).

In [ ]:
def orthobasis_from_filtered(filtered,sv_relative=.15):
    strength=filtered.norm(dim=1)
    good=strength>1e-6
    if not bool(good.any()):return torch.empty(0,784),strength
    normalized=F.normalize(filtered[good],dim=1)
    _,sing,Vh=torch.linalg.svd(normalized,full_matrices=False)
    keep=sing>sv_relative*sing[0]
    return Vh[keep].detach().cpu().contiguous(),strength.detach().cpu()

def score_bank(net,x,index,t):
    f=CACHE/f'score_subspace_{index}_t{t}_k{FILTER_PROBES}_steps{FILTER_LANCZOS}.pt'
    settings=dict(index=int(index),t=int(t),filter_probes=FILTER_PROBES,
                  filter_steps=FILTER_LANCZOS,source_model=MODEL_ID,
                  seed=SEED+index*13+t,cutoff=.1,power=4)
    if f.exists():
        pack=torch.load(f,map_location='cpu',weights_only=True)
        assert pack['settings']==settings
        return pack
    gen=torch.Generator().manual_seed(settings['seed'])
    raw=F.normalize(torch.randn(FILTER_PROBES,784,generator=gen),dim=1).to(DEVICE)
    op=ScoreOperator(net,x.to(DEVICE),t,fd_eps=.04)
    filtered=torch.stack([filter_apply(op,p,FILTER_LANCZOS).detach().cpu() for p in raw])
    basis,strength=orthobasis_from_filtered(filtered)
    pack=dict(settings=settings,basis=basis,strength=strength,
              sigma_ve=float(op.sigma),cost=op.counters(),
              note='Finite-bank score-filtered subspace, not true tangent projector')
    torch.save(pack,f)
    return pack

score_bases={};geometry_rows=[]
for name,model in list(models.items()):
    model.to('cpu')  # gradients are already cached; preserve GPU memory for DDPM
    if DEVICE=='cuda':torch.cuda.empty_cache()
print('Loading frozen DDPM score network:',MODEL_ID)
score_net=model_load(DEVICE)
for j,index in enumerate(active_indices):
    for t in FIXED_TIMESTEPS:
        pack=score_bank(score_net,X[j:j+1],index,t)
        B=pack['basis']; score_bases[(j,t)]=B
        costs=pack['cost']
        geometry_rows.append(dict(index=index,timestep=t,rank=int(len(B)),
            mean_filter_strength=float(pack['strength'].float().mean()),
            total_score_forward_image_evals=int(costs['model_forward_image_evals']),
            total_score_backward_evals=int(costs['model_backward_evals'])))
    if (j+1)%10==0 or j+1==len(active_indices):print(f'Geometry {j+1}/{len(active_indices)} complete')
geom_df=pd.DataFrame(geometry_rows)
geom_df.to_csv(OUT_ROOT/'GEOMETRY_DIAGNOSTICS.csv',index=False)
display(geom_df.groupby('timestep')[['rank','mean_filter_strength']].agg(['mean','min','max']).round(3))
# A second, less rank-limited mechanism: apply the SAME original matrix-function filter
# DIRECTLY to each classifier gradient. This is a soft linear-map approximation,
# NOT a hard orthogonal projection, and not a geometry-independent classifier.
# Its operator is still estimated from the same class-conditional pretrained score net.
direct_filtered={}
effective_ranks={}
direct_cost_rows=[]
for j,index in enumerate(active_indices):
    for t in FIXED_TIMESTEPS:
        base_packet=score_bank(score_net,X[j:j+1],index,t)
        # Hutchinson estimate of tr(H^2), because input probes are unit-normalized:
        # E(||H z||²)=tr(H²)/D. This is an ENERGY-EQUIVALENT rank, not true intrinsic d.
        strength=base_packet['strength'].float()
        raw_erank=float(784*strength.square().mean())
        k_eff=int(np.clip(round(raw_erank),1,min(PCA_NEIGHBORS-1,784)))
        effective_ranks[(j,t)]=k_eff
        f=CACHE/f'direct_filtered_{index}_t{t}_steps{FILTER_LANCZOS}_{RUN_MODE}.pt'
        settings=dict(index=int(index),t=int(t),lanczos=FILTER_LANCZOS,
            classifier_names=list(models),classifier_seeds=[v[2] for v in MODEL_SPECS],
            train_epochs=TRAIN_EPOCHS,train_limit=TRAIN_LIMIT,
            source_model=MODEL_ID,cutoff=.1,power=4)
        if f.exists():
            pack=torch.load(f,map_location='cpu',weights_only=True)
            assert pack['settings']==settings
        else:
            op=ScoreOperator(score_net,X[j:j+1].to(DEVICE),t,fd_eps=.04)
            fg={name:filter_apply(op,grads[name][j],FILTER_LANCZOS).detach().cpu()
                for name in models}
            pack=dict(settings=settings,grads=fg,cost=op.counters())
            torch.save(pack,f)
        direct_filtered[(j,t)]=pack['grads']
        direct_cost_rows.append(dict(index=index,timestep=t,
            energy_equivalent_rank_unclipped=raw_erank,
            matched_control_rank=k_eff,
            capped=(raw_erank>min(PCA_NEIGHBORS-1,784)),
            **pack['cost']))
    if (j+1)%10==0 or j+1==len(active_indices):
        print(f'Direct score-filtered gradients {j+1}/{len(active_indices)} complete')
pd.DataFrame(direct_cost_rows).to_csv(OUT_ROOT/'DIRECT_FILTER_COST_AND_EFFECTIVE_RANK.csv',index=False)
del score_net
if DEVICE=='cuda':torch.cuda.empty_cache()

## 6. Matched-rank random and local-PCA controls

These controls address the main interpretability pitfall: a random subspace can have large cosine similarity simply because dimensionality was reduced. Every control uses exactly the **rank of the score-filtered basis for that image and scale**. PCA neighborhoods come from **MNIST training examples of digit 1**, never from evaluation images or classifier gradients.

In [ ]:
ones_train=train_norm[train_targets==1].flatten(1).float().cpu().contiguous()
train_sq=(ones_train**2).sum(1)

def pca_base(x,rank):
    rank=int(rank)
    if rank<=0:return torch.empty(0,784)
    xf=x.detach().flatten().float().cpu()
    d2=train_sq-2*(ones_train@xf)+xf.square().sum()
    idx=torch.topk(d2,min(PCA_NEIGHBORS,len(d2)),largest=False).indices
    local=ones_train[idx];center=local-local.mean(dim=0,keepdim=True)
    _,_,Vh=torch.linalg.svd(center,full_matrices=False)
    return Vh[:min(rank,len(Vh))].contiguous()

def random_base(rank,index,t,rep):
    if rank<=0:return torch.empty(0,784)
    gen=torch.Generator().manual_seed(SEED+index*137+t*911+rep*10007)
    q,_=torch.linalg.qr(torch.randn(784,rank,generator=gen),mode='reduced')
    return q.T.contiguous()

bases={}
for j,index in enumerate(active_indices):
    for t in FIXED_TIMESTEPS:
        B=score_bases[(j,t)].float()
        k=int(B.shape[0]);bases[(j,t,'score',0)]=B
        bases[(j,t,'pca',0)]=pca_base(X[j],k)
        for rep in range(N_RANDOM_CONTROLS):
            bases[(j,t,'random',rep)]=random_base(k,index,t,rep)

# Projector identities, dimension matching, and orthogonality sanity checks.
for (j,t,kind,rep),B in bases.items():
    assert B.ndim==2 and B.shape[1]==784
    if len(B):
        I=B@B.T
        assert torch.allclose(I,torch.eye(len(B)),atol=1e-3,rtol=1e-3),(j,t,kind)
for j in range(len(active_indices)):
    for t in FIXED_TIMESTEPS:
        k=len(score_bases[(j,t)])
        assert len(bases[(j,t,'pca',0)])==k
        for rep in range(N_RANDOM_CONTROLS):
            assert len(bases[(j,t,'random',rep)])==k
print('PASS all subspace rank-matching and orthogonality checks.')
# Direct-filter comparisons use PCA / random bases matched to the SOFT FILTER'S
# energy-equivalent rank, not the much smaller 8/16-probe bank rank.
direct_controls={}
for j,index in enumerate(active_indices):
    for t in FIXED_TIMESTEPS:
        k=effective_ranks[(j,t)]
        direct_controls[(j,t,'pca_effective',0)]=pca_base(X[j],k)
        for rep in range(N_RANDOM_CONTROLS):
            direct_controls[(j,t,'random_effective',rep)]=random_base(k,index,t,rep+100)
        for kind,rep in [('pca_effective',0)]+[('random_effective',r) for r in range(N_RANDOM_CONTROLS)]:
            B=direct_controls[(j,t,kind,rep)]
            assert len(B)==k and torch.allclose(B@B.T,torch.eye(k),atol=1e-3)
print('PASS direct-filter energy-equivalent-rank control matches.')

## 7. Gradient alignment: full, projected, residual, and magnitude

For a basis `B` with orthonormal rows, `P(g)=BᵀBg`. The **normal residual** is `g−P(g)`.

Report four quantities:

1. `cos_full = cos(g_s,g_t)`;
2. `cos_projected = cos(Pg_s,Pg_t)`;
3. `cos_normal = cos((I−P)g_s,(I−P)g_t)`;
4. `dot_contribution = <Pg_s,Pg_t>/(||g_s||||g_t||)` and `projection_energy_fraction` to prevent misleading claims from small subspaces.

The projected and normal cosine may become undefined if the component is negligible; missing values are recorded rather than forced to 1 or 0.

`score_direct` is the primary measurement (the original filter acting directly on a gradient). `score` is a secondary rank-limited probe-bank projector. For `score_direct`, `cos_normal` refers to the *filter residual*, not an orthogonal normal projection.

In [ ]:
def proj(v,B):
    if len(B)==0:return torch.zeros_like(v)
    return (v@B.T)@B

def cos_safe(a,b,atol=1e-10):
    an=float(a.norm());bn=float(b.norm())
    return float(torch.dot(a,b)/(an*bn)) if an>atol and bn>atol else float('nan')

pairs=list(combinations(list(models),2))
records=[]
for j,index in enumerate(active_indices):
    for t in FIXED_TIMESTEPS:
        for (jj,tt,kind,rep),B in bases.items():
            if jj!=j or tt!=t:continue
            for source,target in pairs:
                a=grads[source][j].float();b=grads[target][j].float()
                pa=proj(a,B);pb=proj(b,B)
                ra=a-pa;rb=b-pb
                na=a.norm().clamp_min(1e-12);nb=b.norm().clamp_min(1e-12)
                records.append(dict(index=index,timestep=t,method=kind,rep=rep,
                    source=source,target=target,rank=len(B),
                    cos_full=cos_safe(a,b),cos_projected=cos_safe(pa,pb),
                    cos_normal=cos_safe(ra,rb),
                    dot_contribution=float(torch.dot(pa,pb)/(na*nb)),
                    source_projection_ratio=float(pa.norm()/na),
                    target_projection_ratio=float(pb.norm()/nb),
                    source_energy_fraction=float(pa.square().sum()/(na*na)),
                    target_energy_fraction=float(pb.square().sum()/(nb*nb)),
                    degenerate_projected=bool(float(pa.norm()/na)<MIN_PROJECTED_RATIO or float(pb.norm()/nb)<MIN_PROJECTED_RATIO)))

# The PRIMARY mechanism test: direct score spectral filter applied to each gradient.
# Keep this distinct from score-filtered PROBE-bank projection above.
for j,index in enumerate(active_indices):
    for t in FIXED_TIMESTEPS:
        for source,target in pairs:
            a=grads[source][j].float();b=grads[target][j].float()
            na=a.norm().clamp_min(1e-12);nb=b.norm().clamp_min(1e-12)
            choices=[('score_direct',0,None)]+[
                (kind,rep,B) for (jj,tt,kind,rep),B in direct_controls.items()
                if jj==j and tt==t]
            for method,rep,B in choices:
                if method=='score_direct':
                    pa=direct_filtered[(j,t)][source].float()
                    pb=direct_filtered[(j,t)][target].float()
                else:
                    pa=proj(a,B);pb=proj(b,B)
                # For the direct filter this residual is NOT an orthogonal normal projector.
                ra=a-pa;rb=b-pb
                records.append(dict(index=index,timestep=t,method=method,rep=rep,
                    source=source,target=target,rank=effective_ranks[(j,t)],
                    cos_full=cos_safe(a,b),cos_projected=cos_safe(pa,pb),
                    cos_normal=cos_safe(ra,rb),
                    dot_contribution=float(torch.dot(pa,pb)/(na*nb)),
                    source_projection_ratio=float(pa.norm()/na),
                    target_projection_ratio=float(pb.norm()/nb),
                    source_energy_fraction=float(pa.square().sum()/(na*na)),
                    target_energy_fraction=float(pb.square().sum()/(nb*nb)),
                    degenerate_projected=bool(float(pa.norm()/na)<MIN_PROJECTED_RATIO or float(pb.norm()/nb)<MIN_PROJECTED_RATIO)))
align=pd.DataFrame(records)
align.to_csv(OUT_ROOT/'PAIRWISE_GRADIENT_ALIGNMENT.csv',index=False)
print('Pairwise rows:',len(align),'| sampled images:',len(active_indices))
display(align.groupby(['timestep','method']).agg(n=('cos_projected','count'),
   cosine=('cos_projected','mean'),dot_contribution=('dot_contribution','mean'),
   source_energy=('source_energy_fraction','mean'),
   degenerate_rate=('degenerate_projected','mean')).round(4))

## 8. One-step target-query-free transfer check

Use a **single precommitted L2 step** along the surrogate's full gradient, score-subspace projection, PCA projection, matched-rank random projections, or the score-subspace normal residual. Clip to `[-1,1]`. The step direction is determined using **only the chosen surrogate's gradient**. Other classifiers' logits are used only afterward for evaluation.

Metrics: mean change in the target's untargeted logit margin, conditional fooling rate (ASR), actual post-clipping L2/L∞ distortion, and number of attacks with negligible projected gradients. Higher projected cosine alone does *not* prove better transfer.

For fairness, the nominal step factor is adjusted **without any classifier queries** until the **achieved post-clipping L2** equals the precommitted budget. Unreachable directions are reported instead of being silently scored as failures.

In [ ]:
def matched_clipped_l2_step(x,d,target_l2,rtol=3e-4):
    """Search input step length so AFTER [-1,1] clipping ||adv-x||_2=target_l2.
    Uses x and surrogate-only direction; no classifier prediction or target feedback.
    Returns None if that geometric direction cannot achieve the requested budget.
    """
    u=d.reshape_as(x)/d.norm().clamp_min(1e-12)
    lo=0.; hi=float(target_l2)
    def step_norm(a):
        return float(((x+a*u).clamp(-1,1)-x).norm())
    for _ in range(18):
        if step_norm(hi)>=target_l2*(1-rtol):break
        hi*=2.
    if step_norm(hi)<target_l2*(1-rtol):return None
    for _ in range(24):
        mid=(lo+hi)/2.
        if step_norm(mid)<target_l2:lo=mid
        else:hi=mid
    adv=(x+hi*u).clamp(-1,1)
    return adv,float(hi)

# Unit test: matched achieved L2 on a clean interior input.
xx=torch.zeros(1,28,28); dd=torch.randn(784,generator=torch.Generator().manual_seed(SEED))
check=matched_clipped_l2_step(xx,dd,ATTACK_EPS_L2)
assert check is not None and abs(float((check[0]-xx).norm())-ATTACK_EPS_L2)<.002
attack_rows=[]
attack_images=[]
attack_meta=[]
for j,index in enumerate(active_indices):
    x=X[j].detach().cpu()
    for source in models:
        gs=grads[source][j].float()
        candidate_dirs=[('full',-1,0,gs)]
        for t in FIXED_TIMESTEPS:
            for kind in ('score','pca','random'):
                nrep=N_RANDOM_CONTROLS if kind=='random' else 1
                for rep in range(nrep):
                    B=bases[(j,t,kind,rep)]
                    candidate_dirs.append((f'{kind}_t{t}',t,rep,proj(gs,B)))
            candidate_dirs.append((f'normal_t{t}',t,0,gs-proj(gs,score_bases[(j,t)])))
            # Direct spectral filtering on each surrogate gradient is the primary test.
            candidate_dirs.append((f'score_direct_t{t}',t,0,direct_filtered[(j,t)][source].float()))
            candidate_dirs.append((f'normal_direct_t{t}',t,0,gs-direct_filtered[(j,t)][source].float()))
            for kind in ('pca_effective','random_effective'):
                nrep=N_RANDOM_CONTROLS if kind=='random_effective' else 1
                for rep in range(nrep):
                    B=direct_controls[(j,t,kind,rep)]
                    candidate_dirs.append((f'{kind}_t{t}',t,rep,proj(gs,B)))

        for method,t,rep,d in candidate_dirs:
            ratio=float(d.norm()/gs.norm().clamp_min(1e-12))
            usable=bool(float(d.norm())>MIN_GRAD_NORM and ratio>=MIN_PROJECTED_RATIO)
            if not usable:
                attack_rows.append(dict(index=index,source=source,method=method,timestep=t,rep=rep,
                   usable=False,projected_norm_ratio=ratio))
                continue
            step=matched_clipped_l2_step(x,d,ATTACK_EPS_L2)
            if step is None:
                attack_rows.append(dict(index=index,source=source,method=method,timestep=t,rep=rep,
                   usable=False,projected_norm_ratio=ratio,reason='direction_cannot_reach_clipped_L2'))
                continue
            adv,preclip_multiplier=step
            pert=(adv-x).flatten()
            attack_images.append(adv)
            attack_meta.append(dict(index=index,source=source,method=method,timestep=t,rep=rep,
                 usable=True,projected_norm_ratio=ratio,
                 actual_L2=float(pert.norm()),actual_Linf=float(pert.abs().max()),
                 preclip_multiplier=preclip_multiplier))
print('Attack candidates generated without target-model outputs:',len(attack_images))
adv_tensor=torch.stack(attack_images)
attack_meta=pd.DataFrame(attack_meta)

# Evaluation ONLY: target classifier forward passes do not inform generation.
all_evals=[]
position={ix:i for i,ix in enumerate(active_indices)}
for target,net in models.items():
    net=net.to(DEVICE).eval()
    target_logits=batch_clean_logits(net,adv_tensor)
    for k,row in attack_meta.iterrows():
        if row.source==target:continue
        true_idx=position[row['index']]
        clean_z=clean_selected[target][true_idx:true_idx+1]
        z=target_logits[k:k+1]
        baseline=float(margin_objective(clean_z,torch.tensor([1])))
        attacked=float(margin_objective(z,torch.tensor([1])))
        d=row.to_dict()
        d.update(target=target,target_margin_change=attacked-baseline,
                 target_fooling=int(z.argmax(1).item()!=1))
        all_evals.append(d)
    net.to('cpu')
    if DEVICE=='cuda':torch.cuda.empty_cache()
transfer=pd.DataFrame(all_evals)
transfer.to_csv(OUT_ROOT/'TRANSFER_ONE_STEP_TARGET_FEEDBACK_FREE_GENERATION.csv',index=False)
pd.DataFrame(attack_rows).to_csv(OUT_ROOT/'UNUSABLE_PROJECTED_DIRECTIONS.csv',index=False)
print('Transfer rows:',len(transfer),'| unusable attempts:',len(attack_rows))
display(transfer.groupby('method').agg(n=('target_fooling','size'),ASR=('target_fooling','mean'),
      margin_increase=('target_margin_change','mean'),L2=('actual_L2','mean'),
      Linf=('actual_Linf','mean')).sort_values('margin_increase',ascending=False).round(4))

## 9. Paired confidence intervals clustered by original image

The same image appears across classifier pairs and random-control replications. Those are **not independent experimental units**. We aggregate to image level before bootstrap resampling. A single split with four trained models provides only preliminary evidence; replication over independently trained model ensembles is necessary for publication-level claims.

In [ ]:
# Image-level summary. Within an image, model pairs and random-basis repetitions are averaged first.
align_img=(align.groupby(['index','timestep','method'],as_index=False)
    [['cos_full','cos_projected','cos_normal','dot_contribution','source_energy_fraction',
      'target_energy_fraction','degenerate_projected']].mean())
align_img.to_csv(OUT_ROOT/'IMAGE_LEVEL_ALIGNMENT.csv',index=False)
transfer_img=(transfer.groupby(['index','method'],as_index=False)
    [['target_fooling','target_margin_change','actual_L2','actual_Linf']].mean())
transfer_img.to_csv(OUT_ROOT/'IMAGE_LEVEL_TRANSFER.csv',index=False)

rng_ci=np.random.default_rng(SEED+3001)
def paired_bootstrap(frame_a,frame_b,metric,id_col='index',reps=BOOTSTRAP_REPS):
    aa=frame_a[[id_col,metric]].rename(columns={metric:'a'})
    bb=frame_b[[id_col,metric]].rename(columns={metric:'b'})
    joined=aa.merge(bb,on=id_col).dropna()
    d=(joined['a']-joined['b']).to_numpy()
    if len(d)<4:return dict(n=len(d),mean_delta=float(np.mean(d)) if len(d) else np.nan,ci_low=np.nan,ci_high=np.nan)
    picks=rng_ci.integers(0,len(d),size=(reps,len(d)))
    boots=d[picks].mean(axis=1)
    return dict(n=len(d),mean_delta=float(d.mean()),ci_low=float(np.quantile(boots,.025)),
                ci_high=float(np.quantile(boots,.975)))

comparisons=[]
for t in FIXED_TIMESTEPS:
    sub=align_img[align_img.timestep==t]
    for control in ('pca','random'):
        for metric in ('cos_projected','dot_contribution','source_energy_fraction'):
            v=paired_bootstrap(sub[sub.method=='score'],sub[sub.method==control],metric)
            comparisons.append(dict(outcome='alignment',t=t,metric=metric,
                comparison=f'score minus {control}',**v))
for control in ('pca_t5','random_t5','full','score_t100'):
    # Legacy finite filtered-probe bank comparison.
    for metric in ('target_fooling','target_margin_change','actual_L2'):
        v=paired_bootstrap(transfer_img[transfer_img.method=='score_t5'],
                transfer_img[transfer_img.method==control],metric)
        comparisons.append(dict(outcome='transfer',t=5,metric=metric,
                 comparison=f'score_t5 minus {control}',**v))
# Primary comparison: score operator applied directly to each gradient.
for t in FIXED_TIMESTEPS:
    sub=align_img[align_img.timestep==t]
    for control in ('pca_effective','random_effective'):
        for metric in ('cos_projected','dot_contribution','source_energy_fraction'):
            v=paired_bootstrap(sub[sub.method=='score_direct'],
                              sub[sub.method==control],metric)
            comparisons.append(dict(outcome='direct_alignment',t=t,metric=metric,
                comparison=f'score_direct minus {control}',**v))
for control in ('pca_effective_t5','random_effective_t5','full','score_direct_t100','score_t5'):
    for metric in ('target_fooling','target_margin_change','actual_L2'):
        v=paired_bootstrap(transfer_img[transfer_img.method=='score_direct_t5'],
                           transfer_img[transfer_img.method==control],metric)
        comparisons.append(dict(outcome='direct_transfer',t=5,metric=metric,
             comparison=f'score_direct_t5 minus {control}',**v))
ci=pd.DataFrame(comparisons)
ci.to_csv(OUT_ROOT/'PAIRED_IMAGE_BOOTSTRAP_CI95.csv',index=False)
display(ci.round(5))

## 10. Proposal-ready figures (actual measurements only)

No illustrative data are included in these charts. Figures display image-level means with **paired, image-clustered confidence intervals** where appropriate. White-box pairwise gradient alignment is a *mechanism diagnostic*; target-query-free one-step attack transfer is a separate outcome.

In [ ]:
def mean_ci(values,boots=1500):
    v=np.asarray(pd.Series(values).dropna(),dtype=float)
    if len(v)==0:return (np.nan,np.nan,np.nan)
    idx=rng_ci.integers(0,len(v),size=(boots,len(v)))
    bs=v[idx].mean(axis=1)
    return (float(v.mean()),float(np.quantile(bs,.025)),float(np.quantile(bs,.975)))

# Figure 1: projected cosine vs matched-rank controls and full-space cosine.
fig,ax=plt.subplots(figsize=(9,4.6))
names=['full','score-direct t5','PCA-eff t5','random-eff t5','score-direct t100']
values=[]
values.append(mean_ci(align_img.query('timestep==5 and method=="score_direct"').cos_full))
for t,kind in [(5,'score_direct'),(5,'pca_effective'),(5,'random_effective'),(100,'score_direct')]:
    values.append(mean_ci(align_img.query('timestep==@t and method==@kind').cos_projected))
for i,(m,lo,hi) in enumerate(values):
    ax.bar(i,m)
    if np.isfinite(lo):ax.errorbar(i,m,yerr=[[max(0,m-lo)],[max(0,hi-m)]],fmt='none',color='black',capsize=4)
ax.axhline(0,color='gray',lw=.7)
ax.set_xticks(range(len(names)),names,rotation=15)
ax.set_ylabel('Mean gradient cosine (95% image-bootstrap CI)')
ax.set_title('Gradient agreement: full space vs score-filtered & controls')
fig.tight_layout();fig.savefig(FIGS/'FIG1_GRADIENT_COSINE.png',dpi=180);plt.show()

# Figure 2: energy captured by small subspaces, which guards against cosine inflation.
fig,ax=plt.subplots(figsize=(9,4.4))
labels=[];heights=[]
for t in FIXED_TIMESTEPS:
    for kind in ['score_direct','pca_effective','random_effective']:
        z=align_img.query('timestep==@t and method==@kind')
        labels.append(f'{kind} t{t}')
        heights.append(float(.5*(z.source_energy_fraction.mean()+z.target_energy_fraction.mean())))
ax.bar(range(len(labels)),heights);ax.set_xticks(range(len(labels)),labels,rotation=20)
ax.set_ylabel('Captured gradient energy fraction')
ax.set_title('How much of each gradient survives projection?')
fig.tight_layout();fig.savefig(FIGS/'FIG2_GRADIENT_ENERGY.png',dpi=180);plt.show()

# Figure 3: matched-budget one-step transfer. Report both margin change and ASR elsewhere.
methods=['full','random_effective_t5','pca_effective_t5','score_direct_t5','score_direct_t100']
fig,ax=plt.subplots(figsize=(9,4.4));vals=[]
for method in methods:vals.append(mean_ci(transfer_img.query('method==@method').target_margin_change))
for i,(m,lo,hi) in enumerate(vals):
    ax.bar(i,m)
    if np.isfinite(lo):ax.errorbar(i,m,yerr=[[max(0,m-lo)],[max(0,hi-m)]],fmt='none',color='black',capsize=4)
ax.axhline(0,color='gray',lw=.8)
ax.set_xticks(range(len(methods)),methods,rotation=15)
ax.set_ylabel('Target margin change (higher = more attack effect)')
ax.set_title('One-step transfer, no target feedback in generation')
fig.tight_layout();fig.savefig(FIGS/'FIG3_TRANSFER_MARGIN.png',dpi=180);plt.show()

# Two separate transfer matrices (surrogate→target), using precommitted models.
for method in ('full','score_direct_t5'):
    pivot=transfer[transfer.method==method].groupby(['source','target']).target_fooling.mean().unstack()
    fig,ax=plt.subplots(figsize=(5.5,4.6))
    mat=pivot.reindex(index=list(models),columns=list(models)).to_numpy(dtype=float)
    im=ax.imshow(mat,vmin=0,vmax=1,cmap='viridis')
    ax.set_xticks(range(len(models)),list(models),rotation=25)
    ax.set_yticks(range(len(models)),list(models))
    ax.set_xlabel('Held-out target');ax.set_ylabel('Surrogate')
    ax.set_title(f'Transfer ASR: {method}')
    for r in range(len(models)):
        for c in range(len(models)):
            if r!=c and np.isfinite(mat[r,c]):ax.text(c,r,f'{mat[r,c]:.2f}',ha='center',va='center',color='white')
    fig.colorbar(im,ax=ax,label='Conditional one-step ASR')
    fig.tight_layout();fig.savefig(FIGS/f'FIG4_TRANSFER_MATRIX_{method}.png',dpi=180);plt.show()

## 11. Interpretation, negative controls, and download

**What would support the hypothesis?** Score-filtered directions should show greater pairwise agreement *than same-rank random and local-PCA controls*, retain non-negligible gradient magnitude, and improve target-margin change or transfer success at **matched achieved distortions**.

**What would reject it?** Improved cosine with negligible gradient energy and no transfer benefit, or comparable effects with random subspaces. Do not call this semantic validity: digit-identity preservation requires separate human or independent semantic evaluation.

The output is a *mechanism study*, not proof that score-derived geometry improves a trained adversarial generator. The calibrated selector is deliberately absent to isolate the geometry hypothesis.

The direct filter compares against PCA/random subspaces of similar estimated **filter energy-equivalent rank**, which is approximate; report both direct-filter and finite-bank results, and do not conflate the soft filter with a hard projection.

In [ ]:
# Report actual results, no unsupported claims.
sel=align_img[align_img.timestep==5]
mean_cos={k:float(sel[sel.method==k].cos_projected.mean()) for k in ['score_direct','pca_effective','random_effective']}
transfer_means={k:float(transfer_img[transfer_img.method==k].target_margin_change.mean()) for k in ['full','score_direct_t5','pca_effective_t5','random_effective_t5']}
summary=f"""# Precommitted MNIST tangent/transfer mechanism pilot

- **Mode:** {RUN_MODE}; initial images: {len(image_indices)}; clean-correct intersection: {len(active_indices)}.
- **Frozen geometry:** class-conditional DDPM epsilon score-Jacobian Hill filter applied DIRECTLY to input gradients; secondary filtered-probe bank. Neither is a validated exact tangent projector.
- **Selection:** precommitted t5/t100; NOT the prior FIT-ranked calibrated noise selector.
- **Models:** {', '.join(models)}; trained on MNIST train split only.
- **Gradient objective:** logsumexp(other logits) - true logit.
- **Direct-filtered gradient cosine at t5:** score={mean_cos['score_direct']:.4f}; PCA={mean_cos['pca_effective']:.4f}; random-matched-rank={mean_cos['random_effective']:.4f}.
- **One-step target margin change (exact matched achieved L2):** full={transfer_means['full']:.4f}; score_t5={transfer_means['score_direct_t5']:.4f}; PCA_t5={transfer_means['pca_effective_t5']:.4f}; random_t5={transfer_means['random_effective_t5']:.4f}.
- **Attack budget:** post-clipping L2 matched at {ATTACK_EPS_L2} in [-1,1] pixel coordinates (within tolerance), by a deterministic geometric line search; achieved Linf differs by direction. No target information is used.
- **Threat-model caveat:** target input gradients used for offline alignment analysis only, never for attack generation or tuning; target outputs are evaluated after candidate generation.
- **Inference scope:** digits 1 only. No measured semantic preservation and no trained generator in this notebook.
- **Inference scope:** one classifier ensemble (two CNN seeds and two other architectures). Bootstrap across image points does not capture variation across model-training seeds.
- **Research claim:** do not generalize positive or negative outcomes from the pilot without replication.
"""
(OUT_ROOT/'PROPOSAL_EVIDENCE_README.md').write_text(summary)
print(summary)
# ZIP only CSV, figures, protocol and readme (not the heavy model/geometry caches)
archive_root=Path('/content/transfer_alignment_archive' if Path('/content').exists() else '/mnt/data/transfer_alignment_archive')
archive_root.mkdir(exist_ok=True)
for f in OUT_ROOT.glob('*.csv'):shutil.copy2(f,archive_root/f.name)
for f in OUT_ROOT.glob('*.json'):shutil.copy2(f,archive_root/f.name)
shutil.copy2(OUT_ROOT/'PROPOSAL_EVIDENCE_README.md',archive_root/'PROPOSAL_EVIDENCE_README.md')
archive_figs=archive_root/'figures';archive_figs.mkdir(exist_ok=True)
for f in FIGS.glob('*.png'):shutil.copy2(f,archive_figs/f.name)
zip_path=shutil.make_archive(str(OUT_ROOT/'MNIST_Tangent_Transfer_Experiment_Results'),'zip',root_dir=str(archive_root))
print('RESULTS FOLDER:',OUT_ROOT)
print('ARCHIVE:',zip_path)
display(FileLink(zip_path))